In [ ]:
import os
import sys
import traceback
import warnings
warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
sys.path.append("../src")

from connection_util import DatabaseConnection
from validation_util import Validation
from savepoints_util import SavePoints
from etl_pipeline_chameleon import run_chameleon_etl_pipeline
from etl_pipeline_project_level import run_project_level_etl_pipeline
from version import create_version
from change_history import record_history
from draft_review import submit_for_review, promote, withdraw_review, review_status, copy_to_project_level

In [51]:
ETL_ROLE = 'gis_update' 

target_username = os.getenv(f'{ETL_ROLE.upper()}_USERNAME')
target_password = os.getenv(f'{ETL_ROLE.upper()}_PASSWORD')


In [52]:
# val_host = os.getenv("VAL_HOST")
# val_database_name = os.getenv("VAL_DATABASE_NAME")
# val_port = os.getenv("VAL_PORT")
# val_username = os.getenv("VAL_USERNAME")
# val_password = os.getenv("VAL_PASSWORD")


# col_host = os.getenv("COL_HOST")
# col_database_name = os.getenv("COL_DATABASE_NAME")
# col_port = os.getenv("COL_PORT")
# col_username = os.getenv("COL_USERNAME")
# col_password = os.getenv("COL_PASSWORD")


# buf_host = os.getenv("BUF_HOST")
# buf_database_name = os.getenv("BUF_DATABASE_NAME")
# buf_port = os.getenv("BUF_PORT")
# buf_username = os.getenv("BUF_USERNAME")
# buf_password = os.getenv("BUF_PASSWORD")

# akw_host = os.getenv("AKW_HOST")
# akw_database_name = os.getenv("AKW_DATABASE_NAME")
# akw_port = os.getenv("AKW_PORT")
# akw_username = os.getenv("AKW_USERNAME")
# akw_password = os.getenv("AKW_PASSWORD")


source_host = os.getenv("SOURCE_HOST")
source_database_name = os.getenv("SOURCE_DATABASE_NAME")
source_port = os.getenv("SOURCE_PORT")
source_username = os.getenv("SOURCE_USERNAME")
source_password = os.getenv("SOURCE_PASSWORD")

chameleon_target_host = os.getenv("CHAMELEON_TARGET_HOST")
chameleon_target_database_name = os.getenv("CHAMELEON_TARGET_DATABASE_NAME")
chameleon_target_port = os.getenv("CHAMELEON_TARGET_PORT")
chameleon_target_username = target_username
chameleon_target_password = target_password

project_level_target_host = os.getenv("PROJECT_LEVEL_TARGET_HOST")
project_level_target_database_name = os.getenv("PROJECT_LEVEL_TARGET_DATABASE_NAME")
project_level_target_port = os.getenv("PROJECT_LEVEL_TARGET_PORT")
project_level_target_username = target_username
project_level_target_password = target_password

In [53]:
# val_conn = DatabaseConnection.connect_to_validation_postgis_database(val_host,val_database_name,val_port,val_username,val_password)
source_conn = DatabaseConnection.connect_to_source_postgis_database(source_host,source_database_name,source_port,source_username,source_password)
chameleon_target_conn= DatabaseConnection.connect_to_target_postgis_database(chameleon_target_host,chameleon_target_database_name,chameleon_target_port,chameleon_target_username,chameleon_target_password)
project_level_target_conn= DatabaseConnection.connect_to_target_postgis_database(project_level_target_host,project_level_target_database_name,project_level_target_port,project_level_target_username,project_level_target_password)



Source database connection successful!
Target database connection successful!
Target database connection successful!


In [ ]:

# SCHEMA_NAMES = {'akwaaba':'akwaaba_schema','buffalo':'buffalo_schema','colobus':'colobus_schema'}
SOURCE_SCHEMA_NAMES = {
    'akwaaba':['akwaaba_schema'],
    'buffalo':['buffalo_schema'],
    'colobus':['colobus_schema']
}

# Final (live) schemas — only changed by promote() after approval
TARGET_SCHEMA_NAMES = {
    'akwaaba':['akwaaba_schema'],
    'buffalo':['buffalo_schema'],
    'colobus':['colobus_schema']
    }

# Draft schemas — where the ETL loads; reviewed before promotion
DRAFT_SCHEMA_NAMES = {project: [schemas[0] + '_draft'] for project, schemas in TARGET_SCHEMA_NAMES.items()}

CLEAR_CACHE = True


## Updating the live layers: run from the terminal, one step at a time

Push, submit and promote are **not** run from this notebook, so "Run All" can never push, lock or promote by accident.
From `etl_pipeline/src`:

```
..\etl_env\Scripts\python.exe layers.py status
..\etl_env\Scripts\python.exe layers.py push buffalo
..\etl_env\Scripts\python.exe layers.py submit buffalo --requested-by "Who asked for it" --note "What changed"
..\etl_env\Scripts\python.exe layers.py promote buffalo --approved-by "Approver 1" "Approver 2"
..\etl_env\Scripts\python.exe layers.py withdraw buffalo --reason "Why"
```

Between submit and promote: dashboard → **View: Pending approval** → **Approval PDF**. Full steps in `RUNBOOK.md`.

In [56]:

# def main_project_level_etl():
#     try:

      

#         # # Extract, transform and load  into Project Level database
#     #    #Akwaaba    
#         # run_project_level_etl_pipeline(SOURCE_SCHEMA_NAMES['akwaaba'][0],TARGET_SCHEMA_NAMES['akwaaba'][0], source_conn, project_level_target_conn)
        
#     #     #Buffalo 
#         run_project_level_etl_pipeline(SOURCE_SCHEMA_NAMES['buffalo'][0],TARGET_SCHEMA_NAMES['buffalo'][0], source_conn, project_level_target_conn)
        
#         # Colobus
#         # run_project_level_etl_pipeline(SOURCE_SCHEMA_NAMES['colobus'][0],TARGET_SCHEMA_NAMES['colobus'][0], source_conn, project_level_target_conn)


#         # RUn Backup
#         # create_version('Updated akwaaba to add protection areas and updated colobus to remove overlaps and add emmanuel update')


#         # Clear cache — use one of:
#         # clear_cache()                                          # all projects
#         # clear_cache('buffalo')                                 # one project
#         # clear_cache(['akwaaba', 'buffalo', 'colobus'])         # multiple projects
        

#         print('All Layers are Loaded')

#     except Exception as e:
#         print("ETL Failed with error message", e)
#         traceback.print_exc()


# main_project_level_etl()
